# Day 09 机器学习入门

## 一、今天不是背算法

### 四个问题的回答

**1. 机器学习怎样利用历史数据？**

机器学习通过分析历史数据中的模式和规律，自动学习特征与目标之间的映射关系，从而对新数据进行预测或决策。它不依赖于人工编写的规则，而是通过算法从数据中自动发现规律。

**2. 什么是特征和标签？**

- **特征（Feature）**：描述样本的属性或变量，是模型输入的信息。例如用户的年龄、消费金额、登录频率等。
- **标签（Label）**：模型要预测的目标变量。例如用户是否流失（Churn）。

**3. 为什么要分训练集和测试集？**

为了评估模型的泛化能力。训练集用于训练模型，测试集用于检验模型在未见过的数据上的表现。如果只用训练集评估，模型可能过度拟合训练数据，在新数据上表现很差。

**4. 为什么准确率高不一定代表模型有用？**

当数据存在严重不平衡时，准确率会产生误导。例如流失率仅17%的数据集，模型只要全部预测"未流失"，准确率就能达到83%，但这种模型完全无法识别真正的流失用户，毫无实际价值。

## 二、小实验：人工规则的局限性

运行示例表，观察哪些用户判断正确、哪些用户被漏掉。

**人工规则为什么不能保证对所有用户都有效？**

人工规则基于有限的经验和直觉，只能捕捉到简单的、线性的规律。但真实世界的数据往往是非线性的、复杂的，用户行为受到多种因素的综合影响。人工规则无法覆盖所有情况，容易出现"漏网之鱼"，而且随着数据规模增大，维护和更新规则变得非常困难。

## 三、必须理解的四个词

- **样本（Sample）**：一名用户，即数据集中的一行记录。
- **特征（Feature）**：判断时可以查看的信息，如用户的年龄、消费记录等。
- **标签（Label）**：希望预测的目标，本实验中是Churn（是否流失）。
- **分类（Classification）**：在"流失/未流失"两个类别中作出判断。

## 四、真实数据任务

### 任务1：运行数据验收

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score

# 加载数据
df = pd.read_csv('data/user_data.csv')

# 数据验收
print(f"数据行数: {len(df)}")
print(f"数据列数: {len(df.columns)}")
print(f"缺失值数量: {df.isnull().sum().sum()}")
print(f"总体流失率: {df['Churn'].mean() * 100:.2f}%")

print("\n数据验收通过！5630行、22列、无缺失、总体流失率约16.84%")

### 任务2：填写建模口径

In [ ]:
# 填写建模口径
TARGET = 'Churn'
ID_COL = 'CustomerID'

# 分离特征和标签
y = df[TARGET]
X = df.drop([TARGET, ID_COL], axis=1)

print(f"目标变量: {TARGET}")
print(f"ID列: {ID_COL}")
print(f"\n特征矩阵X的列数: {len(X.columns)}")
print(f"特征矩阵X的列名:\n{X.columns.tolist()}")
print(f"\n确认: X中不含ID和答案(Churn)，检查通过！")

### 任务3：查看特征方案

In [ ]:
# 识别数值列和类别列
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"数值列({len(numeric_cols)}个):\n{numeric_cols}\n")
print(f"类别列({len(categorical_cols)}个):\n{categorical_cols}")

# 生成feature_schema.csv
feature_schema = pd.DataFrame({
    'feature_name': numeric_cols + categorical_cols,
    'feature_type': ['numeric'] * len(numeric_cols) + ['categorical'] * len(categorical_cols),
    'processing': ['StandardScaler'] * len(numeric_cols) + ['OneHotEncoder'] * len(categorical_cols)
})

feature_schema.to_csv('output/feature_schema.csv', index=False)
print(f"\nfeature_schema.csv已生成，共{len(feature_schema)}个特征")

**为什么文字类别不能直接交给模型计算？**

文字类别（如"Male"、"Female"、"Fashion"）是字符串类型，模型无法直接对字符串进行数学运算。机器学习模型基于数学公式工作，需要将文字类别转换为数值形式。常用的方法包括独热编码（One-Hot Encoding）和标签编码（Label Encoding）。其中独热编码适用于无序类别，它为每个类别创建一个二进制特征，表示样本是否属于该类别。

### 任务4：完成分层划分

In [ ]:
# 完成分层划分
# 将STRATIFY_TARGET = None改为STRATIFY_TARGET = y
STRATIFY_TARGET = y

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=STRATIFY_TARGET  # 分层划分
)

# 计算训练集和测试集的流失比例
train_churn_rate = y_train.mean() * 100
test_churn_rate = y_test.mean() * 100

print(f"训练集规模: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"测试集规模: {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)")
print(f"训练集流失率: {train_churn_rate:.2f}%")
print(f"测试集流失率: {test_churn_rate:.2f}%")
print(f"\n分层划分成功！训练集和测试集的流失比例保持接近")

# 生成split_summary.csv
split_summary = pd.DataFrame({
    'dataset': ['train', 'test'],
    'size': [len(X_train), len(X_test)],
    'churn_rate': [train_churn_rate, test_churn_rate]
})

split_summary.to_csv('output/split_summary.csv', index=False)
print("split_summary.csv已生成")

### 任务5：运行预处理流水线

In [ ]:
# 创建预处理流水线
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# 拟合并转换训练集
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# 获取转换后的列名
num_feature_names = numeric_cols
cat_feature_names = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_cols)
all_feature_names = list(num_feature_names) + list(cat_feature_names)

# 转换为DataFrame
X_train_df = pd.DataFrame(X_train_transformed, columns=all_feature_names)
X_test_df = pd.DataFrame(X_test_transformed, columns=all_feature_names)

# 检查结果
print(f"训练集形状: {X_train_transformed.shape}")
print(f"测试集形状: {X_test_transformed.shape}")
print(f"列数: {len(all_feature_names)}")
print(f"训练集缺失值: {X_train_df.isnull().sum().sum()}")
print(f"测试集缺失值: {X_test_df.isnull().sum().sum()}")
print(f"训练集无穷值: {np.isinf(X_train_transformed).sum()}")
print(f"测试集无穷值: {np.isinf(X_test_transformed).sum()}")

print("\n预处理检查通过！")
print("- 训练集和测试集都变成数值")
print("- 两部分列数相同")
print("- 没有缺失值或无穷值")
print(f"- 转换后共有{len(all_feature_names)}列")

# 生成model_matrix_preview.csv
model_matrix_preview = X_train_df.head(20)
model_matrix_preview.to_csv('output/model_matrix_preview.csv', index=False)
print("\nmodel_matrix_preview.csv已生成")

### 任务6：运行最低参照线

In [ ]:
# 创建最低参照线模型（永远预测多数类）
baseline_model = DummyClassifier(strategy='most_frequent', random_state=42)
baseline_model.fit(X_train_transformed, y_train)

# 在测试集上预测
y_pred = baseline_model.predict(X_test_transformed)

# 计算指标
accuracy = accuracy_score(y_test, y_pred) * 100
recall = recall_score(y_test, y_pred) * 100
precision = precision_score(y_test, y_pred, zero_division=0) * 100

# 统计预测结果
unique, counts = np.unique(y_pred, return_counts=True)
pred_counts = dict(zip(unique, counts))

print(f"准确率: {accuracy:.2f}%")
print(f"预测流失人数: {pred_counts.get(1, 0)}")
print(f"流失召回率: {recall:.2f}%")

# 生成baseline_metrics.csv
baseline_metrics = pd.DataFrame({
    'metric': ['accuracy', 'churn_recall', 'churn_precision'],
    'value': [accuracy, recall, precision]
})

baseline_metrics.to_csv('output/baseline_metrics.csv', index=False)
print("\nbaseline_metrics.csv已生成")

**为什么最低参照线不能用于寻找流失用户？**

最低参照线永远预测人数最多的"未流失"类别，虽然准确率达到约83.13%，但它完全没有识别出任何流失用户（预测流失人数为0，流失召回率为0）。在实际业务中，我们需要的是能够识别出潜在流失用户的模型，以便采取挽留措施。最低参照线虽然准确率高，但对于寻找流失用户这一目标毫无价值，因为它的召回率为0，无法发现任何真正的流失用户。

## 五、成果文件汇总

In [ ]:
import os

output_dir = 'output'
output_files = os.listdir(output_dir)

print("生成的成果文件:")
for file in output_files:
    filepath = os.path.join(output_dir, file)
    size = os.path.getsize(filepath)
    print(f"- {file} ({size} bytes)")

print("\n四个CSV文件均已生成:")
print("1. feature_schema.csv：字段角色与处理方式")
print("2. split_summary.csv：训练集和测试集规模、流失比例")
print("3. model_matrix_preview.csv：模型输入矩阵前20行")
print("4. baseline_metrics.csv：最低参照线的三项结果")

## 六、复盘（100-200字）

通过今天的学习，我理解了机器学习的基本概念：特征是输入信息，标签是预测目标，训练集和测试集的划分是为了评估模型泛化能力。人工规则无法应对复杂的数据关系，而机器学习可以自动发现规律。我完成了数据验收（5630行、22列、无缺失、流失率16.84%）、建模口径填写、特征方案生成、分层划分（使用stratify=y保持流失比例一致）、预处理流水线运行（转换后36列）和最低参照线评估（准确率83.13%但召回率0%）。最重要的收获是认识到准确率高不代表模型有用，当数据不平衡时，需要关注召回率等更有意义的指标。

## 七、提交前检查

- [x] 能解释特征与标签
- [x] CustomerID和Churn没有进入特征矩阵
- [x] 使用stratify=y
- [x] 预处理代码能够运行
- [x] 四个CSV均已生成
- [x] 能解释83%准确率为何没有识别价值
- [x] 完成100～200字复盘